# RDD Dependencies, Lineage Graphs, and Fault Tolerance

**Module**: 02_rdd  
**Notebook**: 04_rdd_dependencies_and_lineage.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: How Spark constructs RDD lineage graphs, the distinction between Narrow and Wide dependencies, and how lineage guarantees fault tolerance.
- **Why does this topic exist?**: Lineage is the foundation of Spark's resilience. Understanding dependencies explains why shuffles exist and how lost data is reconstructed.
- **What problem does it solve?**: Without understanding lineage, developers cannot predict stage boundaries or diagnose recomputation overhead.
- **Where is it used in real-world Data Engineering?**: Explaining lineage recomputation and dependency classifications is a mandatory component of Spark technical interviews.

---

## 2. Narrow vs Wide Dependencies in the Lineage Graph

```
NARROW DEPENDENCIES (Pipelined in 1 Stage, No Network Transfer):
Parent Partitions:   [P0]    [P1]    [P2]
                       |         |         |  (e.g. map, filter)
                       v         v         v
Child Partitions:    [C0]    [C1]    [C2]

WIDE DEPENDENCIES (Shuffle, Stage Boundary, Network Transfer):
Parent Partitions:   [P0]    [P1]    [P2]
                        \   \  /   \  /   /   (e.g. groupByKey, join)
                         v   v v   v v   v
Child Partitions:       [C0]     [C1]
```

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("RDD_Lineage_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

## 3. Inspecting RDD Lineage with `toDebugString()`

In [ ]:
sc = spark.sparkContext

# Build a multi-step RDD pipeline
base_rdd = sc.textFile("/opt/spark-data/retail/orders.csv")
header = base_rdd.first()

cleaned = base_rdd.filter(lambda line: line != header)
mapped = cleaned.map(lambda line: line.split(","))
keyed = mapped.map(lambda cols: (cols[7], 1)) # key by order status
reduced = keyed.reduceByKey(lambda a, b: a + b) # WIDE transformation

print("=== RDD Lineage Graph (toDebugString) ===")
print(reduced.toDebugString().decode('utf-8'))

## 4. Analyzing Stage Boundaries in Lineage Output

Notice the `ShuffledRDD` entry above: that is where Spark places a **Stage Boundary**!

In [ ]:
# Inspect dependencies directly using Python introspection
print(f"reduced RDD dependencies count: {len(reduced.dependencies)}")
for dep in reduced.dependencies:
    print(f"  Dependency Type: {type(dep).__name__}")
    print(f"  Parent RDD:      {dep.rdd}")

---

## Experiment: Verifying Partition Recomputation from Lineage

- **1. What are we testing?**: Demonstrate that evaluating an action on an unpersisted RDD re-executes its entire upstream lineage from disk.
- **2. What do we expect?**: Each action will re-execute the map operations independently.

In [ ]:
call_counter = sc.accumulator(0)

def tracked_map(x):
    global call_counter
    call_counter.add(1)
    return x * 2

rdd_calc = sc.parallelize(range(10), 2).map(tracked_map)

# First Action
print("Action 1 count:", rdd_calc.count())
print(f"Accumulator after Action 1: {call_counter.value} (Expected 10)")

# Second Action on same unpersisted RDD
print("Action 2 sum:  ", rdd_calc.sum())
print(f"Accumulator after Action 2: {call_counter.value} (Expected 20 due to recomputation!)")

**3. What happened?**: The accumulator reached 20 instead of staying at 10.  
**4. Why did it happen?**: Because rdd_calc was not cached, Spark recomputed its entire lineage graph from scratch for Action 2.  
**5. Practical Engineering Lesson**: Uncached RDDs/DataFrames re-evaluate their complete transformation graph on every single action.

---

## Practical Exercises

### Exercise 1 (Beginner): Identify OneToOneDependency
**Problem Statement**: Create an RDD with map and filter, and inspect its dependencies to identify OneToOneDependency.  
**Input / Context**: sc.parallelize(range(10)).map(lambda x: x+1).filter(lambda x: x > 5)

In [ ]:
# Exercise 1: OneToOneDependency

### Exercise 2 (Intermediate): Trace Stage Count in toDebugString
**Problem Statement**: Write a pipeline that performs two distinct shuffles (e.g. reduceByKey followed by sortByKey) and count ShuffledRDD occurrences in toDebugString.  
**Input / Context**: reduced RDD

In [ ]:
# Exercise 2: Count shuffles in lineage

### Exercise 3 (Advanced): Simulate Lineage Recomputation
**Problem Statement**: Build an RDD pipeline from retail/products.csv, take an action, and print lineage depth.  
**Input / Context**: retail/products.csv

In [ ]:
# Exercise 3: Lineage depth

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Identify OneToOneDependency):
```python
r = sc.parallelize(range(10)).map(lambda x: x+1).filter(lambda x: x > 5)
print("Dependencies of filter step:", [type(d).__name__ for d in r.dependencies])
```

#### Solution 2 (Trace Stage Count in toDebugString):
```python
sorted_rdd = reduced.sortByKey()
debug_str = sorted_rdd.toDebugString().decode('utf-8')
print("Lines with ShuffledRDD:")
for line in debug_str.split("\n"):
    if"ShuffledRDD"in line:
        print("->", line)
```

#### Solution 3 (Simulate Lineage Recomputation):
```python
prods = sc.textFile("/opt/spark-data/retail/products.csv").filter(lambda l:"Electronics"in l).map(lambda l: l.split(",")).map(lambda c: (c[2], float(c[3]))).reduceByKey(lambda a, b: a + b)
print(f"Lineage tree:\n{prods.toDebugString().decode('utf-8')}")
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Lineage Graph**: Spark tracks the sequence of transformations so lost partitions can be recomputed deterministically.
- **Narrow Dependency**: 1:1 partition mapping; pipelined within a single stage.
- **Wide Dependency**: Shuffle required; draws stage boundaries in the DAG.

### Essential Technical Terms
- `Lineage`: Deterministic acyclic graph of parent RDD transformations.
- `OneToOneDependency`: Narrow dependency where child partition depends only on 1 parent partition.
- `ShuffleDependency`: Wide dependency where child partition depends on all parent partitions.

### Common Engineering Mistakes
- **Assuming Spark Caches Automatically**: Spark does NOT cache intermediate RDDs; every action recomputes from source unless persisted.
- **Ignoring Lineage Length**: Extremely long iterative lineage graphs can cause JVM StackOverflowError (fix with checkpointing).

### Interview Questions on This Topic
- **Q: How does Spark achieve fault tolerance?**
  - *A*: Through RDD lineage recomputation. If an executor fails, Spark recomputes only the missing partition from its parent dependencies.
- **Q: What is toDebugString()?**
  - *A*: An RDD method that displays the full dependency lineage tree.

### What to Study Next
Open 02_rdd/05_rdd_caching_and_persistence.ipynb to learn caching and memory storage levels.

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')